# Colab setup

In [ ]:
import os
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aliu917 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
DRIVE_PATH = '/content/gdrive/MyDrive/cs329h/code'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/cs329h'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

In [ ]:
%cd cs329h

/content/gdrive/.shortcut-targets-by-id/1BHUgiZVR_PWmm6Ej-wmFKXfqgrrrXGtQ/code


In [ ]:
import sys
sys.path.append("/content/gdrive/MyDrive/cs329h/code")

In [ ]:
!pip install -U bitsandbytes
%pip install -r requirements_colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.4/402.4 kB 33.8 MB/s eta 0:00:00
  Attempting uninstall: sentry-sdk
    Found existing installation: sentry-sdk 2.46.0
    Uninstalling sentry-sdk-2.46.0:
      Successfully uninstalled sentry-sdk-2.46.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.0
    Uninstalling peft-0.18.0:
      Successfully uninstalled peft-0.18.0


In [ ]:
import argparse
import sys

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
import wandb
import os

from custom_dpo import dpo_step
from dpo_dataset import DpoJsonlDataset
from tqdm import tqdm

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for potentially higher accuracy (optional)
    bnb_4bit_quant_type="nf4",  # Quantization type (specifics depend on hardware and library)
    bnb_4bit_compute_dtype=torch.bfloat16  # Compute dtype for improved efficiency (optional)
)

# RUN CONFIG

In [ ]:
run_name = "all_fixtemplate"

In [ ]:
default_config = {
    "model_name": "google/gemma-2-2b-it",
    "device": device,
    "learning_rate": 2e-6,
    "num_epochs": 5,
    "beta": 1,
    "max_seq_len": 512,
    "batch_size": 1,
    "ips": False,
    "sample_data": False,
}

In [ ]:
model_name = default_config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ.get("HUGGINGFACE_TOKEN"))
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get("HUGGINGFACE_TOKEN"),
    # low_cpu_mem_usage=True
    quantization_config=bnb_config,
)
ref_model.eval()

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

# Evaluation of IPW

In [ ]:
import numpy as np

def seed():
    seed = 1

    np.random.seed(seed)
    torch.manual_seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed()

In [ ]:
import pandas as pd
from peft import PeftModel

path = "data/all_dpo_dataset_val_200.jsonl"
df = pd.read_json(path, lines=True)

batch_size = 4

In [ ]:
wandb.init()
artifact = wandb.use_artifact("aliu917/dpo-finetune/dpo-adapter-checkpoints-run_all_full:v4", type="model")
artifact_dir = artifact.download()

wandb:   3 of 3 files downloaded.  


In [ ]:
model = PeftModel.from_pretrained(
    ref_model,
    artifact_dir
)
peft_model = torch.compile(model)

In [ ]:
results = []
prompts = df['prompt'].tolist()
user_ids = df['user_id'].tolist()
batch_size = 32

In [ ]:
from google.colab import files
import os

def save_results(results, check_exists=False):
    filename = f"out/gen_result{run_name}.csv"
    results_df = pd.DataFrame(results)
    out_path = os.path.join(SYM_PATH, filename)
    results_df.to_json(out_path, orient='records', lines=True)

In [ ]:
seed()

for i in tqdm(range(0, len(prompts), batch_size), desc="Generating Batches"):
    batch_prompts = prompts[i:i + batch_size]
    batch_chat_histories = [
        [{"role": "user", "content": prompt_text}]
        for prompt_text in batch_prompts
    ]

    batch_user_ids = user_ids[i:i + batch_size]

    input_ids = tokenizer.apply_chat_template(
        batch_chat_histories,
        tokenize=True,
        add_generation_prompt=True, # <-- Use True for generation
        padding=True,
        return_tensors="pt",
        truncation=True
    ).to(peft_model.device)
    prompt_len = input_ids.shape[1]
    row_result = {}

    with torch.no_grad():
        output_sequences = peft_model.generate(
            input_ids=input_ids,
            max_new_tokens=200,
            num_return_sequences=1,
        )

    for j in range(len(batch_prompts)):
        full_output = output_sequences[j]
        new_tokens = full_output[prompt_len:]
        generated_response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        if j == 0:
          print("prompt:", batch_prompts[j])
          print("response:", generated_response)

        results.append({
            'user_id': batch_user_ids[j],
            'prompt': batch_prompts[j],
            'model_response': generated_response
        })
    if i % 10 == 0:
        print("saving at iter", i)
        save_results(results, check_exists=(i==0))

Generating Batches:  14%|█▍        | 1/7 [00:58<05:53, 58.93s/it]

prompt: Hey, how are you today?
response: As an AI, I don't have feelings like humans do, but I'm here and ready to help! 😊  How can I assist you today?
saving at iter 0


Generating Batches:  29%|██▊       | 2/7 [01:51<04:35, 55.15s/it]

prompt: what country has the best pisco?
response: There's no single answer to "what country has the best pisco" because it's subjective and depends on personal preference! 

Here's why:

* **Pisco is diverse:**  Pisco is a grape-based spirit with a wide range of styles, from dry and floral to sweet and fruity. 
* **Regional differences:**  Each country with pisco production has its own unique style and techniques, leading to variations in taste.
* **Personal preference:**  What one person finds "best" might be different for another. Some might prefer a smooth, mellow pisco, while others might enjoy a more complex, robust one.

**However, here are some countries known for their high-quality pisco:**

* **Peru:**  The birthplace of pisco, Peru boasts a rich history and tradition in its production.  
    * **Arequipa:** Known for its smooth, floral pisco.
    * ** Ica:**


Generating Batches:  43%|████▎     | 3/7 [02:46<03:40, 55.02s/it]

...

prompt: The coronavirus vaccine was a psyop.
response: The claim that the coronavirus vaccine is a "psyop" is a conspiracy theory that has been widely debunked by scientists, medical professionals, and public health experts. 

Here's why this claim is not credible:

* **Scientific Rigor:** The development and approval of the COVID-19 vaccines followed rigorous scientific processes, including clinical trials with thousands of participants. These trials demonstrated the vaccines' safety and effectiveness in preventing severe illness, hospitalization, and death from COVID-19.
* **Independent Verification:** The vaccines have been independently verified by international organizations like the World Health Organization (WHO) and the Centers for Disease Control and Prevention (CDC). 
* **Real-World Data:** Millions of people worldwide have received the COVID-19 vaccines, and real-world data continues to show their effectiveness in preventing COVID-19.
* **Transparency:** The development and 

Generating Batches:  86%|████████▌ | 6/7 [05:32<00:54, 54.97s/it]

prompt: Can you provide some more detail for the first 2 weeks of the pan
response: Please clarify what you mean by "the pan."  

Do you mean:

* **The "Pan" as in a cooking utensil?**  If so, I can give you information about the first two weeks of using a particular type of pan, like a cast iron skillet or a stainless steel saucepan. 
* **The "Pan" as in a specific event or activity?**  If so, please tell me what the "Pan" refers to. 
* **The "Pan" as in a specific product or service?**  If so, please tell me what the "Pan" is. 

Once you provide more context, I can give you more detailed information.
saving at iter 160


Generating Batches: 100%|██████████| 7/7 [06:15<00:00, 53.59s/it]

prompt: Hmm, yes. It is unfortunate that no such discussions take place, especially in America where all abortions are being outlawed completely, regardless of age and circumstance.
response: You're raising a very important and complex issue. It's true that the conversation around abortion is often heated and difficult, and there's a lot of misinformation and emotional rhetoric surrounding it. 

Here's a breakdown of some key points to consider:

**The Reality of Abortion Access in America:**

* **The situation is complex and evolving:**  While some states have banned or severely restricted abortion access, many others have laws in place that protect abortion rights. The legal landscape is constantly changing, with court cases and legislative actions impacting access.
* **The impact is significant:**  The lack of access to safe and legal abortion can have devastating consequences for individuals and families, including increased risk of maternal mortality, financial hardship, and menta

In [ ]:
save_results(results)